# PharmaSense: FDA FAERS Polypharmacy Mining Framework
**Author:** Rishabh Dehariya | Roll No: 23IM10028  
**Supervisor:** Prof. Sarada Prasad Sarmah | IIT Kharagpur  
**Project:** BTP-I — Advanced Pharmacovigilance

This notebook implements the complete **Multi-Gate Polypharmacy Signal Detection** pipeline directly on Google Colab using the OpenFDA API as the data source.

### What this notebook covers:
1. Data Ingestion from OpenFDA API
2. Exploratory Data Analysis (EDA)
3. Disproportionality Analysis (ROR + PRR)
4. FP-Growth Frequent Pattern Mining
5. Logistic Regression (2-drug & 3-drug)
6. XGBoost, Random Forest, LASSO
7. Neural Network (MLP)
8. Temporal Trend Analysis
9. Model Comparison & Visualization

## 0. Install Dependencies

In [ ]:
!pip install -q mlxtend duckdb xgboost lightgbm statsmodels plotly

In [ ]:
import requests
import json
import pandas as pd
import numpy as np
import duckdb
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, roc_curve)
from sklearn.preprocessing import label_binarize
import xgboost as xgb
import lightgbm as lgb
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, apriori, association_rules
warnings.filterwarnings('ignore')
print('✅ All libraries loaded!')

## 1. Data Ingestion from OpenFDA API

In [ ]:
def fetch_faers_page(skip=0, limit=100, year_start='20230101', year_end='20231231'):
    """Fetch a page of FAERS records from the OpenFDA API."""
    url = 'https://api.fda.gov/drug/event.json'
    params = {
        'search': f'receivedate:[{year_start}+TO+{year_end}]',
        'limit': limit,
        'skip': skip
    }
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        return []
    return r.json().get('results', [])

def parse_reports(results):
    """Flatten nested JSON reports into tabular rows."""
    demo_rows, drug_rows, reac_rows = [], [], []
    for r in results:
        caseid = r.get('safetyreportid')
        if not caseid:
            continue
        patient = r.get('patient', {})
        demo_rows.append({
            'caseid': caseid,
            'receive_date': r.get('receivedate'),
            'sex': patient.get('patientsex'),
            'age': patient.get('patientonsetage')
        })
        for d in patient.get('drug', []):
            name = d.get('medicinalproduct', '').strip().upper()
            if name:
                drug_rows.append({'caseid': caseid, 'drugname': name, 'role_cod': d.get('drugcharacterization')})
        for react in patient.get('reaction', []):
            ae = react.get('reactionmeddrapt', '').strip()
            if ae:
                reac_rows.append({'caseid': caseid, 'reaction': ae})
    return demo_rows, drug_rows, reac_rows

# Fetch data
print('Fetching 2023 FAERS records via OpenFDA API...')
all_demo, all_drug, all_reac = [], [], []

for skip in tqdm(range(0, 1000, 100), desc='Fetching pages'):
    results = fetch_faers_page(skip=skip, limit=100)
    if not results:
        break
    d, dr, r = parse_reports(results)
    all_demo.extend(d)
    all_drug.extend(dr)
    all_reac.extend(r)

demo_df = pd.DataFrame(all_demo).drop_duplicates('caseid')
drug_df = pd.DataFrame(all_drug).drop_duplicates()
reac_df = pd.DataFrame(all_reac).drop_duplicates()

# Keep only suspect drugs (role_cod == '1')
drug_df = drug_df[drug_df['role_cod'] == '1'].dropna(subset=['drugname'])

print(f'✅ Reports: {len(demo_df):,} | Drug records: {len(drug_df):,} | Reaction records: {len(reac_df):,}')

## 2. Store in DuckDB & EDA

In [ ]:
con = duckdb.connect(':memory:')
con.register('demo', demo_df)
con.register('drug', drug_df)
con.register('reac', reac_df)

con.execute('CREATE TABLE demo_t AS SELECT * FROM demo')
con.execute('CREATE TABLE drug_t AS SELECT DISTINCT caseid, drugname FROM drug')
con.execute('CREATE TABLE reac_t AS SELECT DISTINCT caseid, reaction FROM reac')

N = con.execute('SELECT COUNT(DISTINCT caseid) FROM demo_t').fetchone()[0]
N_drugs = con.execute('SELECT COUNT(DISTINCT drugname) FROM drug_t').fetchone()[0]
N_aes = con.execute('SELECT COUNT(DISTINCT reaction) FROM reac_t').fetchone()[0]
print(f'Total Reports: {N} | Unique Drugs: {N_drugs} | Unique AEs: {N_aes}')

In [ ]:
# Top 20 drugs
top_drugs = con.execute("""
    SELECT drugname, COUNT(DISTINCT caseid) as reports
    FROM drug_t GROUP BY drugname
    ORDER BY reports DESC LIMIT 20
""").fetchdf()

fig = px.bar(top_drugs, x='reports', y='drugname', orientation='h',
             color='reports', color_continuous_scale='Blues',
             title='Top 20 Most Reported Drugs in FAERS')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='white', height=550)
fig.show()

In [ ]:
# Top 20 adverse events
top_aes = con.execute("""
    SELECT reaction, COUNT(DISTINCT caseid) as reports
    FROM reac_t GROUP BY reaction
    ORDER BY reports DESC LIMIT 20
""").fetchdf()

fig = px.bar(top_aes, x='reports', y='reaction', orientation='h',
             color='reports', color_continuous_scale='Reds',
             title='Top 20 Most Reported Adverse Events in FAERS')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='white', height=550)
fig.show()

## 3. Disproportionality Analysis: ROR + PRR

In [ ]:
def compute_drug_ae_signals(drug_name, top_n=20):
    """Compute ROR and PRR for all AEs associated with a drug."""
    drug_name = drug_name.upper()
    n_drug = con.execute("SELECT COUNT(DISTINCT caseid) FROM drug_t WHERE drugname LIKE ?",
                         [f'%{drug_name}%']).fetchone()[0]
    if n_drug == 0:
        print(f'No reports found for {drug_name}')
        return pd.DataFrame()
    
    df = con.execute(f"""
        SELECT r.reaction,
               COUNT(DISTINCT r.caseid) AS a,
               (SELECT COUNT(DISTINCT caseid) FROM reac_t WHERE reaction = r.reaction) AS n_ae
        FROM reac_t r
        JOIN drug_t d ON r.caseid = d.caseid
        WHERE d.drugname LIKE '%{drug_name}%'
        GROUP BY r.reaction HAVING a >= 2
        ORDER BY a DESC LIMIT {top_n}
    """).fetchdf()
    
    results = []
    for _, row in df.iterrows():
        a, n_ae = row['a'], row['n_ae']
        b = n_drug - a + 0.5
        c = n_ae - a + 0.5
        d = N - n_drug - n_ae + a + 0.5
        a += 0.5
        ror = (a * d) / (b * c)
        se_ror = np.sqrt(1/a + 1/b + 1/c + 1/d)
        ror_l = np.exp(np.log(ror) - 1.96 * se_ror)
        prr = (a/(a+b)) / (c/(c+d))
        se_prr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
        prr_l = np.exp(np.log(prr) - 1.96 * se_prr)
        n = a+b+c+d
        chi2 = (np.abs(a*d - b*c) - n/2)**2 * n / ((a+b)*(c+d)*(a+c)*(b+d))
        results.append({
            'AE': row['reaction'],
            'Reports(a)': int(row['a']),
            'ROR': round(ror, 3),
            'ROR_L95': round(ror_l, 3),
            'PRR': round(prr, 3),
            'PRR_L95': round(prr_l, 3),
            'Chi2': round(chi2, 3),
            'Signal': 'HIGH' if (prr_l >= 2 and chi2 >= 4) else ('MOD' if ror_l > 1 else 'LOW')
        })
    return pd.DataFrame(results)

# Test with Fentanyl
signals = compute_drug_ae_signals('FENTANYL')
print(signals.to_string(index=False))

In [ ]:
# Visualize ROR signals as volcano plot
if not signals.empty:
    fig = px.scatter(signals, x='ROR', y='Chi2', size='Reports(a)', color='Signal',
                     color_discrete_map={'HIGH':'red','MOD':'orange','LOW':'green'},
                     hover_name='AE', text='AE',
                     title='FENTANYL — Adverse Event Signal Map (ROR vs Chi²)',
                     labels={'Chi2':'Chi-Square Statistic'})
    fig.add_hline(y=4, line_dash='dash', annotation_text='Chi²=4 threshold')
    fig.add_vline(x=2, line_dash='dash', annotation_text='ROR=2')
    fig.update_traces(textposition='top center', textfont_size=8)
    fig.update_layout(plot_bgcolor='white', height=500)
    fig.show()

## 4. Frequent Pattern Mining: FP-Growth vs Apriori

In [ ]:
# Build transaction list
transactions_df = con.execute("""
    SELECT caseid, list(drugname) as drugs
    FROM drug_t GROUP BY caseid HAVING count(drugname) >= 2
""").fetchdf()

transactions = transactions_df['drugs'].tolist()
print(f'Transactions with 2+ drugs: {len(transactions):,}')

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions, sparse=True)
df_sparse = pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)

MIN_SUP = max(0.01, 10/len(transactions))
print(f'Using min_support = {MIN_SUP:.4f} ({int(MIN_SUP*len(transactions))} transactions)')

In [ ]:
import time

# FP-Growth
t0 = time.time()
fp_itemsets = fpgrowth(df_sparse, min_support=MIN_SUP, use_colnames=True)
fp_time = time.time() - t0
fp_itemsets['length'] = fp_itemsets['itemsets'].apply(len)
print(f'FP-Growth: {len(fp_itemsets)} itemsets in {fp_time:.2f}s')

# Apriori (for comparison)
t0 = time.time()
ap_itemsets = apriori(df_sparse, min_support=MIN_SUP, use_colnames=True)
ap_time = time.time() - t0
print(f'Apriori: {len(ap_itemsets)} itemsets in {ap_time:.2f}s')

# Compare
comp_df = pd.DataFrame({
    'Algorithm': ['FP-Growth', 'Apriori'],
    'Itemsets Found': [len(fp_itemsets), len(ap_itemsets)],
    'Time (s)': [round(fp_time,3), round(ap_time,3)]
})
print(comp_df.to_string(index=False))

# Triplets
triplets = fp_itemsets[fp_itemsets['length'] == 3].sort_values('support', ascending=False)
print(f'\nFrequent 3-drug triplets: {len(triplets)}')
print(triplets.head(10).to_string(index=False))

## 5. Multi-Gate 3-Drug Polypharmacy Signal Detection

In [ ]:
def ror_for_subset(drugs_list, ae, n_ae):
    conditions = " AND ".join([f"EXISTS (SELECT 1 FROM drug_t dg WHERE dg.caseid=d.caseid AND dg.drugname='{d}')" for d in drugs_list])
    ab = con.execute(f"SELECT COUNT(DISTINCT d.caseid) FROM drug_t d WHERE {conditions}").fetchone()[0]
    a = con.execute(f"""
        SELECT COUNT(DISTINCT r.caseid) FROM reac_t r WHERE r.reaction='{ae}'
        AND {' AND '.join([f"EXISTS (SELECT 1 FROM drug_t d WHERE d.caseid=r.caseid AND d.drugname='{d}')" for d in drugs_list])}
    """).fetchone()[0]
    b, c, d = ab-a+0.5, n_ae-a+0.5, N-ab-n_ae+a+0.5
    a = a+0.5
    ror = (a*d)/(b*c)
    return ror, ab

print('Running Multi-Gate analysis on top triplets...')
results = []

for _, row in tqdm(triplets.head(30).iterrows(), total=min(30,len(triplets))):
    drugs = list(row['itemsets'])
    d1, d2, d3 = drugs[0], drugs[1], drugs[2]
    
    aes = con.execute(f"""
        SELECT r.reaction, COUNT(DISTINCT r.caseid) as a
        FROM reac_t r
        JOIN drug_t g1 ON r.caseid=g1.caseid AND g1.drugname='{d1}'
        JOIN drug_t g2 ON r.caseid=g2.caseid AND g2.drugname='{d2}'
        JOIN drug_t g3 ON r.caseid=g3.caseid AND g3.drugname='{d3}'
        GROUP BY r.reaction HAVING a >= 3
    """).fetchdf()
    
    ror_abc, ab_count = ror_for_subset([d1,d2,d3], '__dummy__', 0) if len(aes)==0 else (None, 0)
    
    for _, ae_row in aes.iterrows():
        ae = ae_row['reaction']
        a_val = ae_row['a']
        n_ae = con.execute("SELECT COUNT(DISTINCT caseid) FROM reac_t WHERE reaction=?", [ae]).fetchone()[0]
        ror_abc, ab = ror_for_subset([d1,d2,d3], ae, n_ae)
        
        if ror_abc <= 1: continue
        
        # Hierarchical check
        lower_rors = [
            ror_for_subset([d1], ae, n_ae)[0],
            ror_for_subset([d2], ae, n_ae)[0],
            ror_for_subset([d3], ae, n_ae)[0],
            ror_for_subset([d1,d2], ae, n_ae)[0],
            ror_for_subset([d1,d3], ae, n_ae)[0],
            ror_for_subset([d2,d3], ae, n_ae)[0],
        ]
        max_lower = max(lower_rors)
        strength = np.log(ror_abc) - np.log(max_lower)
        if strength <= 0: continue
        
        results.append({
            'Drug1': d1, 'Drug2': d2, 'Drug3': d3, 'AE': ae,
            'Reports': int(a_val), 'ROR_ABC': round(ror_abc,3),
            'Max_Lower_ROR': round(max_lower,3),
            'Interaction_Strength': round(strength,4)
        })

signals_df = pd.DataFrame(results).sort_values('Interaction_Strength', ascending=False)
print(f'\n--- 3-Drug Signals Before FDR ---')
print(signals_df.to_string(index=False))

## 6. Logistic Regression — 3-Way Interaction

In [ ]:
def test_3way(d1, d2, d3, ae):
    df = con.execute(f"""
        SELECT d.caseid,
               MAX(CASE WHEN dg.drugname='{d1}' THEN 1 ELSE 0 END) as d1,
               MAX(CASE WHEN dg.drugname='{d2}' THEN 1 ELSE 0 END) as d2,
               MAX(CASE WHEN dg.drugname='{d3}' THEN 1 ELSE 0 END) as d3,
               MAX(CASE WHEN r.reaction='{ae}' THEN 1 ELSE 0 END) as y
        FROM demo_t d
        LEFT JOIN drug_t dg ON d.caseid=dg.caseid AND dg.drugname IN ('{d1}','{d2}','{d3}')
        LEFT JOIN reac_t r ON d.caseid=r.caseid AND r.reaction='{ae}'
        GROUP BY d.caseid
    """).fetchdf()
    df['d1d2'] = df['d1']*df['d2']
    df['d1d3'] = df['d1']*df['d3']
    df['d2d3'] = df['d2']*df['d3']
    df['d1d2d3'] = df['d1']*df['d2']*df['d3']
    if df['y'].sum() < 3: return None, None
    try:
        m = smf.logit('y ~ d1 + d2 + d3 + d1d2 + d1d3 + d2d3 + d1d2d3', data=df).fit(disp=0)
        return m.params.get('d1d2d3'), m.pvalues.get('d1d2d3')
    except:
        return None, None

if not signals_df.empty:
    p_vals = []
    for _, row in signals_df.iterrows():
        beta, pval = test_3way(row['Drug1'], row['Drug2'], row['Drug3'], row['AE'])
        p_vals.append(pval if pval else 1.0)
    
    _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
    signals_df['P_Value'] = p_vals
    signals_df['Q_Value'] = q_vals
    final = signals_df[signals_df['Q_Value'] < 0.05].copy()
    final['Emerging_Signal_Score'] = final['Interaction_Strength'] * final['ROR_ABC']
    print('\n=== FINAL VALIDATED SIGNALS ===')
    print(final.sort_values('Emerging_Signal_Score', ascending=False).to_string(index=False))
else:
    print('No triplet signals found. Try lowering MIN_SUP or fetching more data.')

## 7. Machine Learning Models: Feature Engineering

In [ ]:
TARGET_AE = 'Overdose'  # Change this to any AE of interest
TOP_FEATURES = 50  # Number of top drugs to use as features

top_drug_names = con.execute(f"""
    SELECT drugname FROM drug_t
    GROUP BY drugname ORDER BY COUNT(DISTINCT caseid) DESC LIMIT {TOP_FEATURES}
""").fetchdf()['drugname'].tolist()

# Build binary drug matrix
cols = [f"MAX(CASE WHEN dg.drugname='{d}' THEN 1 ELSE 0 END) AS \"drug_{d}\"" for d in top_drug_names]
feature_query = f"""
    SELECT d.caseid,
           MAX(CASE WHEN r.reaction='{TARGET_AE}' THEN 1 ELSE 0 END) as target,
           {', '.join(cols)}
    FROM demo_t d
    LEFT JOIN drug_t dg ON d.caseid=dg.caseid
    LEFT JOIN reac_t r ON d.caseid=r.caseid AND r.reaction='{TARGET_AE}'
    GROUP BY d.caseid
"""
feat_df = con.execute(feature_query).fetchdf().fillna(0)

X = feat_df.drop(columns=['caseid', 'target'])
y = feat_df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | Positive rate: {y.mean():.3%}')

## 8. Model Training: LASSO, Random Forest, XGBoost, LightGBM, MLP

In [ ]:
def evaluate(name, model, X_te, y_te):
    yp = model.predict(X_te)
    yprob = model.predict_proba(X_te)[:,1] if hasattr(model, 'predict_proba') else yp
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_te, yp),
        'Precision': precision_score(y_te, yp, zero_division=0),
        'Recall': recall_score(y_te, yp, zero_division=0),
        'F1': f1_score(y_te, yp, zero_division=0),
        'ROC_AUC': roc_auc_score(y_te, yprob)
    }, yprob

pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
models_trained = {}
all_results = []
roc_data = {}

# 1. LASSO
print('Training LASSO...')
lasso = LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced', max_iter=2000, C=0.1)
lasso.fit(X_train, y_train)
res, prob = evaluate('LASSO (L1)', lasso, X_test, y_test)
all_results.append(res); roc_data['LASSO'] = prob; models_trained['LASSO'] = lasso

# 2. Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
res, prob = evaluate('Random Forest', rf, X_test, y_test)
all_results.append(res); roc_data['Random Forest'] = prob; models_trained['RF'] = rf

# 3. XGBoost
print('Training XGBoost...')
xgb_m = xgb.XGBClassifier(scale_pos_weight=pos_weight, random_state=42, eval_metric='logloss',
                           n_estimators=200, learning_rate=0.05, max_depth=4, subsample=0.8)
xgb_m.fit(X_train, y_train)
res, prob = evaluate('XGBoost', xgb_m, X_test, y_test)
all_results.append(res); roc_data['XGBoost'] = prob; models_trained['XGB'] = xgb_m

# 4. LightGBM
print('Training LightGBM...')
lgb_m = lgb.LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=200,
                            learning_rate=0.05, verbosity=-1)
lgb_m.fit(X_train, y_train)
res, prob = evaluate('LightGBM', lgb_m, X_test, y_test)
all_results.append(res); roc_data['LightGBM'] = prob; models_trained['LGB'] = lgb_m

# 5. Neural Network (MLP)
print('Training MLP Neural Network...')
mlp = MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=300, random_state=42,
                    early_stopping=True, class_weight=None)
mlp.fit(X_train, y_train)
res, prob = evaluate('MLP Neural Net', mlp, X_test, y_test)
all_results.append(res); roc_data['MLP'] = prob; models_trained['MLP'] = mlp

results_df = pd.DataFrame(all_results)
print('\n=== MODEL COMPARISON ===')
print(results_df.round(4).to_string(index=False))

## 9. Visualization: Model Comparison

In [ ]:
# Radar chart
metrics = ['Accuracy','Precision','Recall','F1','ROC_AUC']
fig = go.Figure()
colors = ['#1565C0','#C62828','#2E7D32','#F57F17','#6A1B9A']
for i, (_, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics] + [row[metrics[0]]]
    fig.add_trace(go.Scatterpolar(r=vals, theta=metrics+[metrics[0]],
                                  fill='toself', name=row['Model'],
                                  line_color=colors[i], opacity=0.7))
fig.update_layout(polar=dict(radialaxis=dict(range=[0,1])),
                  title='Model Performance Radar Chart', height=500)
fig.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(9, 6))
for name, prob in roc_data.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance (XGBoost)
feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': xgb_m.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=False).head(20)
feat_imp['Feature'] = feat_imp['Feature'].str.replace('drug_', '')

fig = px.bar(feat_imp, x='Importance', y='Feature', orientation='h',
             color='Importance', color_continuous_scale='Oranges',
             title='Top 20 Most Important Drugs (XGBoost Feature Importance)',
             labels={'Feature':'Drug'})
fig.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='white', height=550)
fig.show()

## 10. Cross-Validation (K-Fold)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_models = {
    'LASSO': lasso, 'Random Forest': rf, 'XGBoost': xgb_m, 'LightGBM': lgb_m
}
cv_results = []
for name, m in cv_models.items():
    scores = cross_validate(m, X, y, cv=cv, scoring=['f1','roc_auc'], n_jobs=-1)
    cv_results.append({
        'Model': name,
        'F1 Mean': round(scores['test_f1'].mean(), 4),
        'F1 Std': round(scores['test_f1'].std(), 4),
        'AUC Mean': round(scores['test_roc_auc'].mean(), 4),
        'AUC Std': round(scores['test_roc_auc'].std(), 4)
    })

cv_df = pd.DataFrame(cv_results)
print('=== 5-Fold Cross-Validation Results ===')
print(cv_df.to_string(index=False))

## 11. LASSO Feature Selection

In [ ]:
# LASSO selects which drugs are truly predictive
lasso_coef = pd.DataFrame({
    'Drug': X.columns,
    'Coefficient': lasso.coef_[0]
})
selected = lasso_coef[lasso_coef['Coefficient'] != 0].sort_values('Coefficient', ascending=False)
selected['Drug'] = selected['Drug'].str.replace('drug_', '')
print(f'LASSO selected {len(selected)} drugs out of {len(X.columns)}')

fig = px.bar(selected, x='Coefficient', y='Drug', orientation='h',
             color='Coefficient', color_continuous_scale='RdBu_r',
             title=f'LASSO: Selected Drug Coefficients for {TARGET_AE} Prediction')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, plot_bgcolor='white', height=450)
fig.show()

## 12. Summary Table & Export

In [ ]:
print('\n' + '='*60)
print('PHARMASENSE — FAERS POLYPHARMACY ANALYSIS SUMMARY')
print('='*60)
print(f'Total FAERS Reports Analyzed:  {N:,}')
print(f'Unique Drugs:                  {N_drugs:,}')
print(f'Unique Adverse Events:         {N_aes:,}')
print(f'Target Adverse Event:          {TARGET_AE}')
print(f'FP-Growth Triplets Found:      {len(triplets)}')
if not signals_df.empty:
    print(f'3-Drug Signals (pre-FDR):      {len(signals_df)}')
    if 'Q_Value' in signals_df.columns:
        print(f'Validated Signals (q<0.05):    {len(final)}')
print('\nBest Model by ROC-AUC:')
best = results_df.loc[results_df['ROC_AUC'].idxmax()]
print(f'  {best["Model"]} — AUC={best["ROC_AUC"]:.4f}, F1={best["F1"]:.4f}')
print('='*60)

# Export
results_df.to_csv('model_comparison.csv', index=False)
if not signals_df.empty:
    signals_df.to_csv('polypharmacy_signals.csv', index=False)
print('\n✅ Exported model_comparison.csv and polypharmacy_signals.csv')